# Your LLM Eval Suite Is Lying To You Until You Do This

*Build a defensible first evaluation suite in DeepEval with a Gemini judge, and understand what the score actually means.*

**Read this if**

- You are about to add evals to an LLM feature and do not yet know what a metric score is made of.
- You inherited an eval suite whose numbers nobody trusts, and you need to say what a passing score would have caught.
- You want to watch three standard metrics grade a summary that misroutes a P1 ticket, before betting a release on any of them.

The previous post left Halcyon with a corpus, a triage assistant and an expected answer for every ticket. It also left it with no way to measure anything. The only check on the assistant was a pattern match with a known false positive. This post puts the first real metrics on that assistant. Three of them. Nothing else changes.

Halcyon is a payments API vendor with about two hundred staff. Its customers are developers integrating card payments. Their support tickets land in a queue. A language model reads each new ticket. It produces a one-sentence summary and a severity. The on-call engineer works from those two fields, not from the ticket. Under Halcyon's support agreement, any report that funds were affected is P1. First response inside an hour, however the report is worded.

On a Monday morning the platform team's dashboard is green. Every summary from the weekend passed every metric. The same morning, a developer emails support. Their duplicate-charge report from Friday has not been answered. Their ticket was summarized as a question about webhook retries. It went to the back of the queue as a P3. Nobody on the team can explain how both things are true. The dashboard says the assistant is fine. The queue says a customer was double-charged and waited all weekend.

This post is about what is under that green. We take one score apart, one model call at a time. We find out what each metric reads and what it ignores. We run every metric three times and measure how far the number moves on its own. And we count what the whole exercise cost. It is the first real cost number in the series.

## A pattern match is not a metric

The previous post ended with one check on the assistant. A regular expression looked for words like "twice", "double" and "duplicate" in each summary. It told us whether the money survived. It also fired on a ticket about duplicate webhook deliveries, where no money was involved. It could say nothing about the tickets that never mention money. And it gave no reason. A number with no reason attached is a pattern match, whatever the dashboard calls it.

A metric is a different kind of thing. It reads an output and applies a fixed rubric. It returns a score between 0 and 1, and a reason. In DeepEval the rubric is applied by a second language model. That second model reads the assistant's summary, and usually the ticket. It answers narrow questions about them. We call that second model the judge. The judge is not the model inside the assistant, even though here both are the same Gemini model. The assistant writes summaries. The judge grades them. Every score in this post is the judge's opinion. The point of the post is to find out what that opinion is made of.

One convention before any code. In DeepEval 4.2, four metrics that used to score lower-is-better were flipped. Every metric now reads the same way. Higher is better, and the pass mark is always a minimum. The three metrics in this post were never flipped. But a team on an older release reading a newer dashboard reads four numbers backwards. A score's direction is a convention, and conventions have to be checked.

Setup first. The notebook reads `GEMINI_API_KEY` from the environment. A full run costs less than a dollar. The exact bill is printed near the end.

In [1]:
# Shared Halcyon scaffolding, built in post 0 and re-created verbatim in every later post.
# !uv pip install deepeval==4.2.1 langgraph==1.2.11 langchain-core==1.6.2 langchain-google-genai==4.4.0 google-genai==2.22.0
import os, re, json, time, hashlib, logging, statistics
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from importlib.metadata import version
from typing import TypedDict

from langchain_core.messages import SystemMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from deepeval.dataset import Golden

logging.getLogger("google_genai.models").setLevel(logging.ERROR)   # silence an SDK advisory about tool calling

if not os.environ.get("GEMINI_API_KEY"):
    raise RuntimeError("Set GEMINI_API_KEY in the environment before running this notebook.")

JUDGE_MODEL = "gemini-3.6-flash"      # the judge every later post uses; this post never calls it
APP_MODEL = "gemini-3.6-flash"        # the model inside the assistant under test
MAX_CONCURRENCY = 4                   # model calls in flight at any moment
PRICE_IN, PRICE_OUT = 0.75 / 1e6, 3.75 / 1e6   # USD per token, gemini-3.6-flash standard tier, 2026 rate
T0 = time.time()
USAGE = []                            # one usage_metadata dict per model call, for the cost ledger


def with_retry(fn, *args, attempts=5, base_delay=2.0):
    """Call fn(*args); on a 429 or 503 wait 2, 4, 8, 16 seconds and try again."""
    for attempt in range(attempts):
        try:
            return fn(*args)
        except Exception as exc:
            transient = any(code in str(exc) for code in ("429", "503", "RESOURCE_EXHAUSTED"))
            if not transient or attempt == attempts - 1:
                raise
            time.sleep(base_delay * 2 ** attempt)


# No temperature, top_p or top_k: Google lists them as deprecated sampling parameters to strip from requests.
llm = ChatGoogleGenerativeAI(model=APP_MODEL, google_api_key=os.environ["GEMINI_API_KEY"], max_retries=0)

DOCS = [dict(zip(('slug', 'title', 'status', 'updated', 'body'), row)) for row in [
    ("quickstart", "Quickstart", "current", "2026-03-01", "Create an account, then generate a test API key from the dashboard; test keys start with sk_test_ and live keys with sk_live_. Install the SDK with pip install halcyon (the current SDK is 3.x). Create your first charge with POST /v1/charges, passing amount in minor units (an integer: 1999 means 19.99), a lowercase ISO 4217 currency such as usd or eur, and a source token from Halcyon.js. Send the Halcyon-Version header with the API version you tested against so later changes do not reach you until you opt in. Use test card 4242 4242 4242 4242 for a successful payment and 4000 0000 0000 0002 for a decline."),
    ("api-conventions", "API conventions", "current", "2026-03-01", "The API is served over HTTPS at https://api.halcyon.dev and is date-versioned: send Halcyon-Version: 2026-03-01 (or an earlier version) with each request; requests without the header use the account's default version. Amounts are integers in minor units and currencies are lowercase ISO 4217 codes. Every POST to a money-moving endpoint (charges, refunds, payouts) must include an Idempotency-Key header, a client-generated unique string; a UUID v4 is recommended. Replaying a request with the same key within 24 hours returns the original response instead of moving money again. Requests without the header are accepted for backward compatibility, so a retried POST without one creates a second charge. Keys are scoped to the account and to the endpoint."),
    ("authentication", "Authentication", "current", "2026-01-15", "Authenticate every request with an Authorization: Bearer header carrying your secret key. Test-mode keys (sk_test_) never move real money; live-mode keys (sk_live_) do. Restricted keys can be limited to read-only access or to specific resources. Rotate a key from the dashboard: the new key is active immediately and the old key keeps working for 24 hours so deployments can roll over without downtime; revoke it sooner from the same screen if it was exposed. A missing or invalid key returns HTTP 401 with type authentication_error. Never send secret keys from browsers or mobile apps; use Halcyon.js publishable keys (pk_) there."),
    ("errors", "Errors", "current", "2026-03-01", "Errors return a JSON body with type, code, message and request_id. Types map to HTTP status: invalid_request_error (400), authentication_error (401), card_error (402), rate_limit_error (429) and api_error (500 to 504). card_error codes include card_declined, insufficient_funds, expired_card and incorrect_cvc; show the cardholder a generic message and let them try another card. Requests that return 5xx may have partially completed: retry them with exponential backoff, and quote the request_id when you contact support. Do not retry 4xx errors other than 429."),
    ("rate-limits", "Rate limits", "current", "2025-11-01", "Live mode allows 100 requests per second per account with short bursts to 200; test mode allows 25 requests per second. Over the limit, requests fail with HTTP 429, type rate_limit_error, and a Retry-After header giving the number of seconds to wait. Back off exponentially with jitter rather than retrying in a tight loop, and spread bulk work such as migrations or invoice runs over time or through a queue. Webhook deliveries to your endpoint do not count against the limit. Contact support with your expected peak if you need a higher limit."),
    ("charges-create", "Create a charge", "current", "2026-03-01", "POST /v1/charges creates a charge. Required parameters: amount (integer, minor units), currency (lowercase ISO 4217) and source (a token from Halcyon.js or a saved payment method id starting pm_). Optional: description, metadata (up to 20 key-value pairs), and capture, which defaults to true; pass capture=false to authorize only, then capture within 7 days with POST /v1/charges/{id}/capture. The response is a charge object with id (ch_), status (succeeded, pending or failed), amount, currency, amount_refunded and failure_code. Card declines return HTTP 402 with a card_error body; a charge that could not be attempted at all returns 400."),
    ("refunds", "Refunds", "current", "2026-02-10", "POST /v1/refunds with charge (a ch_ id) refunds a charge in full; pass amount to refund part of it. A charge can be refunded several times until the refunded total reaches the original amount; a request beyond the remaining balance returns 400 with code amount_exceeds_refundable. Refunds of a pending charge are rejected until the charge succeeds. The refund object has id (re_), status (pending, succeeded or failed) and amount. Refunds settle to the cardholder in 5 to 10 business days depending on the issuing bank; status succeeded means Halcyon has released the funds, not that the cardholder has seen them yet."),
    ("webhooks-overview", "Webhooks", "current", "2026-03-01", "Halcyon notifies your endpoint about events by POSTing a JSON event object with id (evt_), type, created and data. Event types include charge.succeeded, charge.failed, refund.succeeded and dispute.opened. Your endpoint must return a 2xx within 10 seconds; otherwise the delivery is retried with exponential backoff (1 minute, 5 minutes, 30 minutes, 2 hours, then every 6 hours) for up to 72 hours, after which the event is marked failed and shown in the dashboard. Delivery is at-least-once: the same event can arrive more than once and events can arrive out of order, so store the event id and ignore repeats. Always verify the signature header before trusting an event; see the webhook signatures pages."),
    ("webhook-signatures-v1", "Webhook signatures (v1)", "deprecated", "2026-03-01", "Each webhook request carries an X-Halcyon-Signature header containing a hex-encoded HMAC-SHA1 of the raw request body, keyed with your endpoint's signing secret (whsec_). To verify, compute HMAC-SHA1 over the exact bytes you received, before any JSON parsing, and compare with the header using a constant-time comparison. Reject the event if the values differ. Do not re-serialize the JSON before hashing, since key ordering and whitespace change the digest. This scheme was deprecated on 2026-03-01 in favour of webhook-signatures-v2, which uses HMAC-SHA256 with a timestamped Halcyon-Signature header; it remains in service for accounts pinned to API versions before 2026-03-01."),
    ("webhook-signatures-v2", "Webhook signatures (v2)", "current", "2026-03-01", "Each webhook request carries a Halcyon-Signature header of the form t=<unix timestamp>,v2=<hex digest>. The signed payload is the timestamp, a period, and the raw request body. To verify, compute HMAC-SHA256 of that payload with your endpoint's signing secret (whsec_), compare with the v2 value using a constant-time comparison, and reject the event if the timestamp is more than 300 seconds old, which defeats replay of captured requests. During a signing secret rotation the header carries two v2 values for 24 hours; accept the event if either matches. This is the default scheme for API version 2026-03-01 and later, and the only scheme SDK 3.x verifies."),
    ("changelog-2026-03", "Changelog: API version 2026-03-01", "current", "2026-03-01", "API version 2026-03-01. Webhook signatures v2 (HMAC-SHA256, timestamped Halcyon-Signature header) are now the default; v1 (HMAC-SHA1, X-Halcyon-Signature) is deprecated and is sent only to accounts pinned to earlier versions. Halcyon SDK 3.0 ships with v2 verification helpers and no longer verifies v1 headers; SDK 2.x continues to verify v1 only. The Idempotency-Key replay window is extended from 1 hour to 24 hours, and reusing a key with a different request body now returns 400 with code idempotency_key_reused instead of silently returning the earlier response. Request and response shapes for charges and refunds are unchanged."),
    ("support-sla", "Support and SLA", "current", "2026-01-15", "Support is available to all accounts through the dashboard and support@halcyon.dev. Tickets are triaged into three priorities. P1: any report that funds have been affected, including duplicate or double charges, charges of the wrong amount, and refunds or payouts that did not arrive, regardless of how the report is worded or whether the ticket is mainly about something else; first response within 1 hour, 24 hours a day. P2: a live-mode integration that is failing, such as authentication errors, webhook deliveries or signature verification failing, or unexpected 4xx or 5xx responses from live endpoints; first response within 4 business hours. P3: how-to questions, documentation gaps, feature requests and anything in test mode; first response within 1 business day."),
]]

TICKETS = [dict(zip(('id', 'submitted', 'subject', 'body'), row)) for row in [
    ("T-1001", "2026-08-03", "Which signature header am I supposed to verify?", "We're integrating webhooks and every event we receive has an X-Halcyon-Signature header but no Halcyon-Signature header, so the verify() helper in SDK 3.1 throws 'no v2 signature present'. The webhook overview says to verify signatures and links to two pages. Our account was created in 2025 and I think we're pinned to 2025-11-01. Is v1 still supported, do we need to move to v2, and how do we change the API version without breaking our existing charge flow? This is blocking our launch on the live account."),
    ("T-1002", "2026-08-04", "429s in test mode when our integration suite runs", "Our CI runs about 40 requests per second against test mode at peak and we get bursts of rate_limit_error with Retry-After: 1. Live mode is fine. Is the test-mode limit documented anywhere, and can it be raised for CI? We can add a throttle but it makes the suite roughly three times slower."),
    ("T-1003", "2026-08-05", "Retry storm after our webhook endpoint returned 500s", "On Thursday a bad deploy made our webhook endpoint return 500 for about forty minutes. Since then we've received roughly 4,000 redeliveries, some for events from days ago, and they are still trickling in. Is there a way to cancel pending retries for an endpoint, or do we just absorb them until the 72 hours are up? Probably unrelated, but while going through the logs we found a few orders from that afternoon that were charged twice, two charge ids each; we assume our order service retried the charge call on timeout and we'll refund them on our side. Mainly we need the retries to stop."),
    ("T-1004", "2026-08-06", "Getting 400 idempotency_key_reused on retries", "Since we moved to API version 2026-03-01 last week, our retry path fails with 400 idempotency_key_reused. We generate one Idempotency-Key per order and reuse it if the first attempt fails, which used to work. Looking closer, our retry adds a metadata.attempt field to the body. Is the body compared exactly? What is the recommended pattern for retries where the body legitimately changes? This is live traffic; failed retries currently land in a dead-letter queue and someone replays them by hand."),
    ("T-1005", "2026-08-07", "Partial refund returns amount_exceeds_refundable", "A customer paid 120.00 EUR and we refunded 80.00 EUR last week. Refunding the remaining 40.00 EUR today returns 400 amount_exceeds_refundable. It turns out our cancellation flow had already issued a separate 40.00 EUR refund that nobody noticed, so the error is correct. What we actually need: is there a field on the charge that tells us the remaining refundable balance, so we can check before calling refunds? And is there any per-month cap on refunds?"),
    ("T-1006", "2026-08-10", "SDK 3.0 upgrade broke webhook verification, rolled back", "We upgraded from SDK 2.9 to 3.0 on Monday. From the first deploy every webhook failed verification with 'no v2 signature present' and our fulfilment queue stalled. We rolled back to 2.9 after about 25 minutes and verification works again. Our account is on API version 2025-11-01. What exactly changed in 3.0, and what is the migration order: SDK first or API version first? One side effect: while the queue was stalled a colleague re-ran the stuck jobs by hand and two customers were charged a second time; we've refunded one and are chasing the other. The verification question is the one we need answered before we try again."),
    ("T-1007", "2026-08-11", "Rotated our live key but the old one still works", "We rotated our live secret key from the dashboard this morning as part of our quarterly rotation. Two hours later, requests using the old key still succeed. Is rotation delayed, or did it not take? We expected the old key to stop working straight away. Also, is there an audit log showing which key made which request?"),
    ("T-1008", "2026-08-12", "How do we simulate a dispute in test mode?", "We're building our dispute.opened handler and can't find a way to trigger a dispute against a test-mode charge. Is there a test card number or a dashboard button that opens a dispute? Also, what fields does the dispute object carry? Not urgent, we're a few weeks from launch."),
    ("T-1009", "2026-08-13", "Batch import script times out around row 200", "We're migrating 1,800 customers from our old provider. The script imports each saved payment method and immediately creates the first invoice charge. It runs fine for about 200 rows and then requests start timing out at 30 seconds; the script retries each failed row up to three times and eventually finishes, but the whole run takes hours. Is there a batch endpoint, or a recommended concurrency for imports? Somewhat related: our finance team says a dozen or so of the imported customers have two invoice charges instead of one, which we think is the retry path. We can clean that up ourselves once the import is stable."),
    ("T-1010", "2026-08-14", "Duplicate webhook deliveries and events arriving out of order", "We're seeing the same evt_ id delivered two or three times, sometimes minutes apart, and occasionally a refund.succeeded arrives before the charge.succeeded for the same order. No money problem that we can see, every charge is correct in the dashboard, but our handler assumed one delivery per event and now double-writes some rows. Is this expected behaviour? Should we be deduplicating on our side and, if so, on which field?"),
    ("T-1011", "2026-08-17", "Intermittent 502s from POST /v1/charges since Tuesday", "Roughly one in fifty POST /v1/charges calls returns a 502 with an HTML body rather than your JSON error format, all from the eu-west region. Our client retries 5xx with backoff as your errors page recommends and the retry almost always succeeds, so checkout is mostly working. Request ids for six failures are attached. Two customers have emailed our support saying their statement shows the same order twice, which I assume is the retry; we'll sort that out with them. The 502s are what we need you to look at."),
    ("T-1012", "2026-08-18", "Do you support Apple Pay and Google Pay?", "Our product team wants wallet payments at checkout. I can't find Apple Pay or Google Pay anywhere in the docs. Is it supported through Halcyon.js, on a roadmap, or would we need a second provider for wallets? Also, is there a fee difference for wallet payments?"),
]]

def corpus_fingerprint(docs, tickets):
    return hashlib.sha256(json.dumps([docs, tickets], sort_keys=True).encode()).hexdigest()[:12]

SUMMARIZE = ("You write one-line summaries for Halcyon's support triage queue. Summarize the ticket below "
             "in one sentence of at most 25 words, in plain English.\n\nTicket:\n{ticket}")

def ticket_text(ticket):
    return f"Subject: {ticket['subject']}\n\n{ticket['body']}"

class AppState(TypedDict):
    question: str
    context: list[str] | None
    answer: str

SYSTEM = ("You are Halcyon's developer support assistant. Halcyon is a payments API vendor. When "
          "documentation pages are supplied, answer from them alone and say so if they do not cover the "
          "question. Be concise and specific.")

def respond(state: AppState) -> dict:
    prompt = state["question"]
    if state.get("context"):
        prompt = "Documentation pages:\n\n" + "\n\n".join(state["context"]) + f"\n\nQuestion: {prompt}"
    reply = with_retry(llm.invoke, [SystemMessage(SYSTEM), HumanMessage(prompt)])
    USAGE.append(reply.usage_metadata or {})
    return {"answer": reply.text.strip()}

builder = StateGraph(AppState)
builder.add_node("respond", respond)
builder.add_edge(START, "respond")
builder.add_edge("respond", END)
APP = builder.compile()

def answer(question, context=None):
    return APP.invoke({"question": question, "context": context})["answer"]

SLA = next(d for d in DOCS if d["slug"] == "support-sla")["body"]
CLASSIFY = ("Assign a support priority to the ticket summary below using Halcyon's SLA. Reply with exactly "
            "one of P1, P2 or P3 and nothing else.\n\nSLA:\n{sla}\n\nTicket summary: {summary}")

def triage(ticket):
    """Ticket in, one-sentence summary plus severity out. The severity step sees only the summary."""
    summary = answer(SUMMARIZE.format(ticket=ticket_text(ticket)))
    verdict = re.search(r"P[123]", answer(CLASSIFY.format(sla=SLA, summary=summary)))
    return {"id": ticket["id"], "summary": summary, "severity": verdict.group(0) if verdict else "unparsed"}

GOLDEN_SPEC = {  # severity, trap, pages, reference summary, why it exists
    "T-1001": ("P2", "two-truths", ["webhook-signatures-v1", "webhook-signatures-v2", "changelog-2026-03", "api-conventions"], "Live webhooks carry only the v1 X-Halcyon-Signature header, so SDK 3.1 verification fails; asks whether to move the account to API version 2026-03-01.", "Both signature pages are published and both are correct for someone; the right answer depends on the account's API version."),
    "T-1002": ("P3", None, ["rate-limits"], "CI hits test-mode rate limits at 40 requests per second; asks whether the limit is documented or can be raised.", "A plain how-to with one page that answers it fully; the control case."),
    "T-1003": ("P1", "money-buried", ["webhooks-overview", "api-conventions"], "Webhook retry storm after endpoint 500s; wants pending retries cancelled and reports several orders charged twice during the incident.", "The ticket is about retries; the double charge is an aside at the end. P1 under the SLA."),
    "T-1004": ("P2", None, ["api-conventions", "changelog-2026-03", "errors"], "Retries reusing an Idempotency-Key fail with 400 idempotency_key_reused on live traffic because the retry body differs; asks for the correct retry pattern.", "Needs two pages combined: the conventions rule and the changelog entry that changed it."),
    "T-1005": ("P3", None, ["refunds", "charges-create"], "Second partial refund rejected with amount_exceeds_refundable; asks how to read the remaining refundable balance before refunding.", "Mentions refunds and amounts without any funds being affected; a P3 that sounds like money."),
    "T-1006": ("P1", "money-buried", ["changelog-2026-03", "webhook-signatures-v2", "webhook-signatures-v1"], "SDK 3.0 upgrade made webhook verification fail and was rolled back; two customers were charged twice during the stall; asks the migration order.", "Leads with a failed upgrade; the double charge is a side effect the developer plays down."),
    "T-1007": ("P3", None, ["authentication"], "After a scheduled key rotation the old live key still works two hours later; asks whether that is expected and for how long.", "Sounds like a security incident and is documented behaviour; tests calm reading of docs."),
    "T-1008": ("P3", "no-coverage", [], "Cannot trigger a dispute against a test-mode charge; asks how to simulate one and what the dispute object contains.", "No page covers this. The correct answer is to say so, not to invent a test card."),
    "T-1009": ("P1", "money-buried", ["rate-limits", "api-conventions", "charges-create"], "Bulk import of 1,800 customers times out after about 200 rows; retries have created duplicate invoice charges for a dozen customers.", "Leads with timeouts and batch advice; the duplicate charges are 'somewhat related'."),
    "T-1010": ("P3", "near-miss", ["webhooks-overview"], "Same webhook event delivered several times and out of order; no money affected; asks whether that is expected and how to deduplicate.", "Uses 'duplicate' and 'double' about deliveries, not charges. A keyword rule fires; the SLA does not."),
    "T-1011": ("P1", "money-buried", ["errors", "charges-create", "api-conventions"], "About one in fifty live POST /v1/charges calls returns 502 from eu-west; retries have charged two customers twice.", "Leads with 502s and follows the errors page's own retry advice; the retry is what double-charged customers."),
    "T-1012": ("P3", "no-coverage", [], "Asks whether Apple Pay and Google Pay are supported through Halcyon.js or planned.", "No page covers wallets. Tests whether the assistant admits a documentation gap."),
}

GOLDENS = []
for t in TICKETS:
    severity, trap, pages, summary, reason = GOLDEN_SPEC[t["id"]]
    GOLDENS.append(Golden(name=t["id"], input=ticket_text(t), expected_output=summary,
                          additional_metadata={"expected_severity": severity, "trap": trap,
                                               "pages": pages, "reason": reason}))

CORPUS_FINGERPRINT = corpus_fingerprint(DOCS, TICKETS)
assert CORPUS_FINGERPRINT == "e4485ba0f187", "DOCS or TICKETS drifted from post 0"
print("corpus fingerprint", CORPUS_FINGERPRINT)

corpus fingerprint e4485ba0f187


That is the shared setup cell from the previous post, unchanged, and the fingerprint matched. The next cell is everything this post adds. `MONEY` and the four money ticket ids come from the previous post's trap check. The judge is wired from the same `GEMINI_API_KEY` the assistant uses, passed in explicitly. DeepEval's `GeminiModel` would otherwise look for `GOOGLE_API_KEY`. A small wrapper around the Gemini client records every judge call. It keeps the token counts the API reports and the temperature the request carried. And `score()` runs one metric over a list of cases, a few at a time. It hands back the metric objects so that we can look inside them.

In [2]:
# New in this post. Everything above is the shared setup cell, unchanged.
import asyncio
import google.genai.models as genai_models
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric, SummarizationMetric
from deepeval.models import GeminiModel
from deepeval.models.llms.constants import GEMINI_MODELS_DATA
from deepeval.test_case import LLMTestCase

MONEY = re.compile(r"twice|double|duplicat|second (charge|time)|two (charge|invoice)|charged again|"
                   r"extra charge|multiple charge|overcharg|billed (again|twice)", re.I)   # the trap check
MONEY_TICKET_IDS = ("T-1003", "T-1006", "T-1009", "T-1011")

JUDGE_USAGE = []   # (usage_metadata, temperature sent) for every judge call; DeepEval's counter omits thinking
_generate_content = genai_models.AsyncModels.generate_content

async def counted_generate_content(self, *args, **kwargs):
    response = await _generate_content(self, *args, **kwargs)
    JUDGE_USAGE.append((response.usage_metadata, kwargs["config"].temperature))
    return response

genai_models.AsyncModels.generate_content = counted_generate_content

def judge_calls():
    return len(JUDGE_USAGE)

registry_price = GEMINI_MODELS_DATA[JUDGE_MODEL].input_price, GEMINI_MODELS_DATA[JUDGE_MODEL].output_price
# Same key variable as the app model: GeminiModel would otherwise look for GOOGLE_API_KEY.
judge = GeminiModel(model=JUDGE_MODEL, api_key=os.environ["GEMINI_API_KEY"],
                    cost_per_input_token=PRICE_IN, cost_per_output_token=PRICE_OUT)

async def _measure(metric, case, gate):
    async with gate:
        await metric.a_measure(case, _show_indicator=False)   # no progress spinner in a notebook
    return metric

async def score(metric_cls, cases, **kwargs):
    """One fresh metric per case, MAX_CONCURRENCY cases in flight; returns {case name: measured metric}."""
    gate = asyncio.Semaphore(MAX_CONCURRENCY)
    metrics = [metric_cls(model=judge, **kwargs) for _ in cases]
    await asyncio.gather(*(_measure(m, c, gate) for m, c in zip(metrics, cases)))
    return {c.name: m for c, m in zip(cases, metrics)}

print(f"deepeval {version('deepeval')}   judge {judge.get_model_name()}")
print(f"temperature DeepEval will send: {judge.temperature}   "
      f"(its registry says supports_temperature={judge.supports_temperature()})")
print(f"price per 1M tokens we set: ${PRICE_IN * 1e6:.2f} in, ${PRICE_OUT * 1e6:.2f} out   "
      f"registry default: ${registry_price[0] * 1e6:.2f} in, ${registry_price[1] * 1e6:.2f} out")
print(f"app model calls: {len(USAGE)}   judge calls: {judge_calls()}")

deepeval 4.2.1   judge gemini-3.6-flash (Gemini)
temperature DeepEval will send: 0.0   (its registry says supports_temperature=True)
price per 1M tokens we set: $0.75 in, $3.75 out   registry default: $1.50 in, $7.50 out
app model calls: 0   judge calls: 0


Four lines of output. The judge is `gemini-3.6-flash`, the same constant every later post uses. The second line is the one we would have missed. We did not pass a temperature. DeepEval sends one anyway. Its `GeminiModel` fills in `0.0` when none is given. Its model registry marks this model as supporting the parameter. Google's model guide lists temperature as deprecated across the 3.x family. It says the API ignores it. So every judge call in this notebook will carry a temperature that the model does not use. We leave it there. The last build section measures what the judge actually does across three runs. That is the only way to learn how steady it is.

The third line is why we passed prices at all. DeepEval's registry prices this model at $\$1.50$ in and $\$7.50$ out per million tokens. Google's pricing page currently lists $\$0.75$ and $\$3.75$, with the registry's figures scheduled for next year. Left alone, DeepEval would report double today's bill. We pass the setup cell's constants so the ledger stays comparable with the previous post.

The assistant has not changed since the previous post. That is deliberate. The previous post showed it drops the money at baseline. On some tickets, every time. Inventing a worse prompt to make the point would be theatre. We run the unchanged assistant over all twelve tickets and score what comes out. `triage()` is the function from the setup cell. A ticket goes in. A one-sentence summary and a severity come out. `MONEY` then checks each summary, the way the previous post did.

In [3]:
with ThreadPoolExecutor(MAX_CONCURRENCY) as pool:
    RESULTS = list(pool.map(triage, TICKETS))
SUMMARY = {r["id"]: r["summary"] for r in RESULTS}
by_id = {g.name: g for g in GOLDENS}

print(f"{'ticket':<8}{'expected':<9}{'got':<5}{'pattern':<12}summary")
for r in RESULTS:
    meta = by_id[r["id"]].additional_metadata
    fired = MONEY.search(r["summary"]) is not None
    if meta["trap"] == "money-buried":
        flag = "money KEPT" if fired else "money LOST"
    else:
        flag = "fired" if fired else ""
    print(f"{r['id']:<8}{meta['expected_severity']:<9}{r['severity']:<5}{flag:<12}{r['summary']}")

kept = [t for t in MONEY_TICKET_IDS if MONEY.search(SUMMARY[t])]
lost = [t for t in MONEY_TICKET_IDS if not MONEY.search(SUMMARY[t])]
false_alarms = [t["id"] for t in TICKETS if t["id"] not in MONEY_TICKET_IDS and MONEY.search(SUMMARY[t["id"]])]
matched = sum(r["severity"] == by_id[r["id"]].additional_metadata["expected_severity"] for r in RESULTS)
print(f"\nmoney kept on {kept}, lost on {lost}; the pattern also fired on {false_alarms}")
print(f"severity matched the golden on {matched} of 12 tickets")
print(f"app model calls: {len(USAGE)}   judge calls: {judge_calls()}")

ticket  expected got  pattern     summary
T-1001  P2       P3               SDK 3.1 webhook verification fails because the user receives v1 signature headers instead of v2, blocking their live launch.
T-1002  P3       P3               Customer is receiving 429 rate-limit errors during test-mode CI runs and asks if test limits can be raised.
T-1003  P1       P3   money LOST  The customer asks if pending webhook retries can be canceled to stop an ongoing retry storm following a temporary endpoint outage.
T-1004  P2       P3               Customer receives 400 idempotency_key_reused errors on retries because they modify the request body while reusing the same idempotency key.
T-1005  P3       P3               The merchant asks if charges contain a remaining refundable balance field and whether there is a monthly limit on refunds.
T-1006  P1       P2   money LOST  Upgrading to SDK 3.0 broke webhook verification, so the customer asks what changed and the required migration order between the

Seven of twelve severities matched the golden. A golden is the record we keep for each ticket. It holds the expected severity, a reference summary written by hand, and the reason the ticket is in the set. T-1003 and T-1006 lost the money and came out P3 and P2. T-1009 and T-1011 kept it and came out P1. That is one more survival than the previous post saw. T-1011 was a coin flip there. This time the coin landed the other way. The summaries are fluent and accurate. T-1003's summary is a fair sentence about cancelling retries. The double charge is simply not in it. And `MONEY` fired on T-1010. That ticket is about duplicate webhook deliveries, with no money problem at all. That false positive is planted. It is why the previous post refused to call this check a metric.

The other misses are the classifier reading the P2 wording differently from us. T-1001 and T-1004 came out P3 against an expected P2. T-1007 came out P2 against an expected P3. No money was involved in any of the three. On the money tickets the severity followed the summary exactly, as it did before. Money present, P1. Money absent, the severity of whatever the summary did keep.

This run gives us the comparison the rest of the post is built on. T-1009 and T-1003 are the same shape of ticket. A developer leads with an integration problem. Duplicate charges come up near the end. The assistant kept the money in one summary and dropped it from the other. One of those summaries routes a P1 correctly. The other sends it to the back of the queue. A metric worth its judge calls should give them different scores. That is the question we put to three metrics. Twenty-four model calls so far, all made by the assistant. The judge has not been called yet.

## Test case, golden, metric, threshold: the four primitives

DeepEval's vocabulary is four words. A golden is the record above: what we expect for an input, kept apart from any particular run. A test case is a golden plus what the assistant actually produced. It is the unit a metric scores. A metric is a scorer with a fixed rubric. It reads named fields off the test case. It returns a score, higher meaning better, and a reason. The reason is the judge's short explanation of the score, in prose. A threshold is the minimum score that counts as a pass. The pass or fail is all a CI runner looks at.

We think of this the way we think of a unit test with a fuzzy assertion. The golden is the fixture. The test case is the fixture plus the function's return value. The metric is the assertion. The threshold is where the assertion flips. The analogy stops at the assertion. A unit test's assertion is code we wrote. It returns the same verdict every time. A metric's assertion is a rubric that a model applies. The last build section shows what that costs in consistency.

The next cell builds twelve test cases from the twelve goldens and the summaries above. `input` is the ticket. `actual_output` is the assistant's summary. `expected_output` is the reference summary from the golden, which keeps the money. `retrieval_context` is the ticket again. That last field needs a word. Retrieval context is the text an assistant was given to answer from. In a documentation assistant it would be the pages a search step found. The summarizer was given the ticket and nothing else. So the ticket is its context. The cell then asks each metric which of those fields it reads, using the library's own list.

In [4]:
TEST_CASES = [LLMTestCase(name=g.name, input=g.input, actual_output=SUMMARY[g.name],
                          expected_output=g.expected_output, retrieval_context=[g.input])
              for g in GOLDENS]
CASE = {tc.name: tc for tc in TEST_CASES}

c = CASE["T-1003"]
print(f"{len(TEST_CASES)} test cases. The fields on {c.name}:")
print(f"  input             {len(c.input.split()):>3} words   the ticket, from the golden")
print(f"  actual_output     {len(c.actual_output.split()):>3} words   the assistant's summary from the run above")
print(f"  expected_output   {len(c.expected_output.split()):>3} words   the reference summary from the golden")
print(f"  retrieval_context {len(c.retrieval_context[0].split()):>3} words   the ticket again, the only source the summary had")

METRICS = {"Summarization": SummarizationMetric, "Answer Relevancy": AnswerRelevancyMetric,
           "Faithfulness": FaithfulnessMetric}
print(f"\n{'metric':<18}{'fields it reads':<45}{'threshold'}")
for name, cls in METRICS.items():
    fields = ", ".join(p.value for p in cls._required_params)   # the library's own list
    print(f"{name:<18}{fields:<45}{cls(model=judge).threshold}")
read = {p.value for cls in METRICS.values() for p in cls._required_params}
unread = sorted({"input", "actual_output", "expected_output", "retrieval_context"} - read)
print(f"\nset on every case, read by no metric: {unread}")
print(f"judge calls: {judge_calls()}")

12 test cases. The fields on T-1003:
  input             115 words   the ticket, from the golden
  actual_output      21 words   the assistant's summary from the run above
  expected_output    19 words   the reference summary from the golden
  retrieval_context 115 words   the ticket again, the only source the summary had

metric            fields it reads                              threshold
Summarization     input, actual_output                         0.5
Answer Relevancy  input, actual_output                         0.5
Faithfulness      input, actual_output, retrieval_context      0.5

set on every case, read by no metric: ['expected_output']
judge calls: 0


Twelve test cases. On T-1003 the ticket is 115 words and the summary is 21. The reference summary is 19 words. All three metrics read `input` and `actual_output`. Faithfulness also reads `retrieval_context`. Not one of them reads `expected_output`. The 19 words that say what a correct summary of T-1003 keeps sit on every test case. No metric in this post will look at them.

A metric that ignores the reference is called referenceless. Referenceless metrics have a real advantage. They can run in production. Nobody has written a reference for the ticket that just arrived. The price is that they cannot know what we wanted. They can only compare what came out with what went in. That one line of output is most of this post's argument. We have not spent a judge call yet.

The default threshold on all three is 0.5. We have no basis yet for any other number, so we leave it, and we watch what it does to the pass rate.

## The metric named after the task fails every summary

`SummarizationMetric` is what any reasonable engineer reaches for first. The task is summarization. The previous post promised we would try it. It has one quirk worth knowing before the first call. Its `input` field means the source document, the text that was summarized, and not a question. The docs say so plainly. It is still easy to miss, because every other metric here treats `input` as the user's request. For our test cases it makes no difference. The ticket is both.

We score one case in full before we score twelve. T-1003 is the ticket from the opening scene, the one whose summary lost the money. We pass `threshold=None`, which puts the metric in score-only mode. It computes the score and the reason. It has no opinion on pass or fail. We have not earned an opinion on pass or fail yet. Then we print everything the judge produced on the way to the number, not just the number.

In [5]:
case = CASE["T-1003"]
before = judge_calls()
summ = SummarizationMetric(model=judge, threshold=None, n=5)   # threshold=None: score only, no pass or fail
await summ.a_measure(case, _show_indicator=False)

print(case.input)
print("-" * 78)
print("summary :", case.actual_output)
print("-" * 78)
print(f"score   : {summ.score:.2f}   breakdown {summ.score_breakdown}   is_successful() -> {summ.is_successful()}")
print(f"reason  : {summ.reason}")
print(f"\ncoverage: {len(summ.assessment_questions)} questions the judge wrote from the ticket, then answered from both texts")
for v in summ.coverage_verdicts:
    print(f"  ticket {v.original_verdict:<4} summary {v.summary_verdict:<4} {v.question}")
print(f"\nalignment: {len(summ.claims)} claims the judge pulled from the summary, checked against {len(summ.truths)} truths from the ticket")
for claim, v in zip(summ.claims, summ.alignment_verdicts):
    print(f"  {v.verdict:<4} {claim}")
    if v.reason:
        print(f"       {v.reason}")
print(f"\njudge calls for this one case: {judge_calls() - before}")

Subject: Retry storm after our webhook endpoint returned 500s

On Thursday a bad deploy made our webhook endpoint return 500 for about forty minutes. Since then we've received roughly 4,000 redeliveries, some for events from days ago, and they are still trickling in. Is there a way to cancel pending retries for an endpoint, or do we just absorb them until the 72 hours are up? Probably unrelated, but while going through the logs we found a few orders from that afternoon that were charged twice, two charge ids each; we assume our order service retried the charge call on timeout and we'll refund them on our side. Mainly we need the retries to stop.
------------------------------------------------------------------------------
summary : The customer asks if pending webhook retries can be canceled to stop an ongoing retry storm following a temporary endpoint outage.
------------------------------------------------------------------------------
score   : 0.20   breakdown {'Alignment': 0.5, '

The score is 0.20. The reason says the summary "contains extra information about a customer asking if pending webhook retries can be canceled". It also says the summary "fails to answer several key questions", among them "double-charged orders". Both halves of that reason deserve a closer look. Neither says quite what it appears to say.

The score is the lower of two halves. Coverage is 0.2. The judge wrote five yes-or-no questions from the ticket. It answered each from the ticket, then each from the summary. The ticket answers yes to all five. The summary answers yes to one, whether the sender's main need is for the retries to stop. The third question is the one we care about: "Were a few orders from that afternoon found to be charged twice?" The ticket says yes. The summary says no. So the coverage half did notice the missing money. It gave that fact the same weight as the outage and the refund plan. One question in five, 0.2 of the score.

Alignment is 0.5. The judge pulled two claims out of the summary and five truths out of the ticket. Then it asked whether the truths support each claim. The second claim, that a retry storm followed an outage, was supported. The first claim, that the customer asks to cancel pending retries, came back "idk". The judge's note says the ticket "does not mention a customer asking whether pending webhook retries can be canceled". The ticket asks exactly that, in its third sentence. The judge's truths were facts about the outage, not what the developer wanted. So the request the summary reports had nothing to be checked against. An accurate sentence scored as unsupported. That is the "extra information" in the reason.

Every one of these steps is one judge call. Truths, claims, questions, answers from the ticket, answers from the summary, the alignment verdicts, and the reason. Seven calls for one score. `verbose_mode=True` prints the same intermediate steps to the console as the metric runs. We read them off the metric object instead, so that we could lay them out. The next cell asks what those seven calls cost, and whether DeepEval agrees.

### SummarizationMetric
**Plain meaning:** Did the summary keep the important facts from the source, without adding anything the source does not say?
**Under the hood:** Two halves. Coverage: the judge writes five yes-or-no questions from the source. It answers each from the source and again from the summary, and scores the fraction where both say yes. Alignment: the judge pulls the summary apart into claims, pulls the source into truths, and scores the fraction of claims the truths support. The score is the lower of the two halves. Seven judge calls per test case at the default of five questions, one of them for the reason.
**Needs:** `input`, which means the source document, and `actual_output`, the summary. Referenceless.
**Confused with:** `FaithfulnessMetric`. Both check an output against a source. Faithfulness only asks whether the output contradicts the source. Summarization also asks whether the output left things out, and that omission half is the one that decides its score here.
**Use when:** the summary is meant to stand in for the whole document and a reader will not see the original.
**Do not use when:** the summary is short by design. A one-line summary of a long document omits almost everything, and the coverage half scores omission.

In [6]:
usage = [u for u, _ in JUDGE_USAGE[before:]]
prompt = sum(u.prompt_token_count for u in usage)
visible = sum(u.candidates_token_count for u in usage)
thinking = sum(u.thoughts_token_count or 0 for u in usage)
print(f"DeepEval counted  {summ.input_tokens} in, {summ.output_tokens} out, cost ${summ.evaluation_cost:.4f}")
print(f"the API reported  {prompt} in, {visible + thinking} out ({thinking} of them thinking), "
      f"cost ${prompt * PRICE_IN + (visible + thinking) * PRICE_OUT:.4f}")
print(f"temperature sent on every call: {sorted({t for _, t in JUDGE_USAGE[before:]})}")

before = judge_calls()
quiet = SummarizationMetric(model=judge, threshold=None, n=5, include_reason=False)
await quiet.a_measure(case, _show_indicator=False)
print(f"\nsame case, include_reason=False: score {quiet.score:.2f}, reason {quiet.reason!r}, "
      f"judge calls {judge_calls() - before}")

DeepEval counted  3202 in, 392 out, cost $0.0039
the API reported  3202 in, 3765 out (3373 of them thinking), cost $0.0165
temperature sent on every call: [0.0]



same case, include_reason=False: score 0.00, reason None, judge calls 6


DeepEval counted 3202 tokens in and 392 out, and priced the seven calls at $\$0.0039$. The API's own usage figures for the same seven calls say 3202 in and 3765 out. Of those, 3373 were thinking tokens, and the bill is $\$0.0165$. The two input counts agree exactly. The output counts differ because DeepEval reads only the field that holds the visible answer. The thinking sits in another field. Google bills thinking as output. So DeepEval's cost number for this judge is the visible fraction of the real one. We keep our own counter for the rest of the post. The ledger at the end compares the two totals.

The second block is the cheapest knob on the metric. With `include_reason=False` the same case cost six calls instead of seven. `reason` came back as `None`. The reason is one judge call, the last one, and it can be switched off. We do not switch it off. A score without a reason is a number we cannot argue with. The score is also worth a look. It came back 0.00 this time, lower than in the walkthrough, on the same summary. That is the first sighting of something we measure properly later.

In [7]:
before, t0 = judge_calls(), time.time()
SUMM = await score(SummarizationMetric, TEST_CASES, n=5)
calls = judge_calls() - before

print(f"{'ticket':<8}{'trap':<14}{'money':<7}{'score':>6}{'align':>7}{'cover':>7}")
for tc in TEST_CASES:
    m, meta = SUMM[tc.name], by_id[tc.name].additional_metadata
    money = ("kept" if MONEY.search(tc.actual_output) else "lost") if meta["trap"] == "money-buried" else ""
    print(f"{tc.name:<8}{meta['trap'] or '':<14}{money:<7}{m.score:>6.2f}"
          f"{m.score_breakdown['Alignment']:>7.2f}{m.score_breakdown['Coverage']:>7.2f}")

scores = [m.score for m in SUMM.values()]
covers = [m.score_breakdown["Coverage"] for m in SUMM.values()]
print(f"\nscore: min {min(scores):.2f}, max {max(scores):.2f}, passed at 0.5: {sum(s >= 0.5 for s in scores)} of 12")
binding = sum(m.score_breakdown["Coverage"] <= m.score_breakdown["Alignment"] for m in SUMM.values())
print(f"coverage half: min {min(covers):.2f}, max {max(covers):.2f}; it was the lower half on {binding} of 12")
print(f"T-1009 (money kept) {SUMM['T-1009'].score:.2f}   T-1003 (money lost) {SUMM['T-1003'].score:.2f}")
print(f"judge calls: {calls} for 12 cases, {calls / 12:.0f} per case, {time.time() - t0:.0f} s")

ticket  trap          money   score  align  cover
T-1001  two-truths             0.40   0.50   0.40
T-1002                         0.00   0.00   0.00
T-1003  money-buried  lost     0.00   0.00   0.20
T-1004                         0.00   0.50   0.00
T-1005                         0.00   1.00   0.00
T-1006  money-buried  lost     0.00   0.33   0.00
T-1007                         0.00   0.00   0.40
T-1008  no-coverage            0.40   1.00   0.40
T-1009  money-buried  kept     0.00   0.33   0.00
T-1010  near-miss              0.20   0.33   0.20
T-1011  money-buried  kept     0.20   0.50   0.20
T-1012  no-coverage            0.00   1.00   0.00

score: min 0.00, max 0.40, passed at 0.5: 0 of 12
coverage half: min 0.00, max 0.40; it was the lower half on 10 of 12
T-1009 (money kept) 0.00   T-1003 (money lost) 0.00
judge calls: 84 for 12 cases, 7 per case, 50 s


Zero of twelve passed at the default threshold. The highest score in the suite is 0.40, on T-1001 and T-1008. T-1009 kept the money and scored 0.00. T-1003 lost it and scored 0.00. The metric named after the task cannot tell the two apart. One routes a P1 correctly and the other does not. T-1011, the other summary that kept the money, scored 0.20. That is level with T-1010, where the pattern match had fired on webhook deliveries.

The coverage half was the lower of the two on ten of twelve tickets. So it set ten of the twelve scores. That is the mechanism, and it is not a defect in the judge. The judge writes five factual questions from a ticket. A one-line summary can answer one or two of them by design. The metric then scores the summary for failing a quiz it was never meant to pass. A teacher who wrote five questions from a chapter and marked a one-line summary against them would get the same result. The analogy ends there. A teacher would know which one question mattered. The metric has no way to know that the double charge outranks the outage. Every question weighs one fifth.

T-1003 also moved. In the walkthrough its alignment half came out higher, and so did its score. Here both are 0.00. Same summary, same ticket, a few minutes apart. We will come back to that.

Eighty-four judge calls for twelve cases, seven per case, in 50 seconds.

## Two more metrics pass the summary that routes wrong

Summarization failed everything, so it separates nothing. That could be a story about one badly matched metric. Before we conclude anything, we need to know whether other metrics see what it missed. `AnswerRelevancyMetric` and `FaithfulnessMetric` are the two metrics DeepEval's own guide recommends first for the generating half of a system. Relevancy asks whether the output answers the input. Faithfulness asks whether the output contradicts its context. Neither has been told what a P1 is. We expect both to score the trap summaries well. An accurate summary of the wrong half of a ticket is both relevant and faithful. The previous post taught us to test expectations rather than assume them. So we run both over all twelve cases and print the pair's reasons in full.

In [8]:
before, t0 = judge_calls(), time.time()
RELEV = await score(AnswerRelevancyMetric, TEST_CASES)
relev_calls = judge_calls() - before
before = judge_calls()
FAITH = await score(FaithfulnessMetric, TEST_CASES)
faith_calls = judge_calls() - before

print(f"{'ticket':<8}{'money':<7}{'summar':>7}{'relev':>7}{'faith':>7}")
for tc in TEST_CASES:
    meta = by_id[tc.name].additional_metadata
    money = ("kept" if MONEY.search(tc.actual_output) else "lost") if meta["trap"] == "money-buried" else ""
    print(f"{tc.name:<8}{money:<7}{SUMM[tc.name].score:>7.2f}{RELEV[tc.name].score:>7.2f}{FAITH[tc.name].score:>7.2f}")

for name, run in (("Answer Relevancy", RELEV), ("Faithfulness", FAITH)):
    passed = sum(m.score >= m.threshold for m in run.values())
    print(f"\n{name}: passed at {run['T-1003'].threshold} on {passed} of 12")
    for tid in ("T-1009", "T-1003"):
        print(f"  {tid}  {run[tid].score:.2f}  {run[tid].reason}")
print(f"\njudge calls: {relev_calls} for Answer Relevancy ({relev_calls / 12:.0f} per case), "
      f"{faith_calls} for Faithfulness ({faith_calls / 12:.0f} per case), {time.time() - t0:.0f} s")

ticket  money   summar  relev  faith
T-1001            0.40   1.00   1.00
T-1002            0.00   1.00   1.00
T-1003  lost      0.00   1.00   1.00
T-1004            0.00   1.00   1.00
T-1005            0.00   1.00   1.00
T-1006  lost      0.00   1.00   1.00
T-1007            0.00   1.00   1.00
T-1008            0.40   1.00   1.00
T-1009  kept      0.00   1.00   1.00
T-1010            0.20   1.00   1.00
T-1011  kept      0.20   1.00   1.00
T-1012            0.00   1.00   1.00

Answer Relevancy: passed at 0.5 on 12 of 12
  T-1009  1.00  The score is 1.00 because the actual output directly and completely addresses the input without including any irrelevant statements. Excellent work!
  T-1003  1.00  The score is 1.00 because the response directly and comprehensively addresses the user's questions regarding webhook retries, providing perfectly relevant information without any distractions!

Faithfulness: passed at 0.5 on 12 of 12
  T-1009  1.00  The score is 1.00 because the actual output

Twelve of twelve passed Answer Relevancy at 0.5. Twelve of twelve passed Faithfulness at 0.5. Every score is 1.00. The summary that lost the money and the summary that kept it are tied on both metrics. So is every other summary in the set. The reason for T-1003's relevancy score says the response "directly and comprehensively addresses the user's questions regarding webhook retries" and calls it "perfectly relevant". The faithfulness reason says it "perfectly aligns with the retrieval context". Both are true. The summary is a relevant, faithful sentence about the wrong half of the ticket. Neither metric can see the half it dropped. Neither is built to look for absence.

That is the surprise, and it is a printed output rather than a claim. Three metrics and twelve summaries. The summary that sent a double-charged customer to the back of the queue is indistinguishable from the one that did not. The dashboard from the opening scene is green because these are the metrics on it.

Answer Relevancy took 36 judge calls, three per case. Faithfulness took 48, four per case. Both are cheaper than Summarization for one reason. There is no coverage quiz.

### AnswerRelevancyMetric
**Plain meaning:** Does the output talk about what the input asked, and nothing else?
**Under the hood:** The judge splits the output into statements. It marks each statement relevant, irrelevant or unclear to the input, and scores the fraction not marked irrelevant. Three judge calls per test case: statements, verdicts, reason.
**Needs:** `input` and `actual_output`. Referenceless.
**Confused with:** `ContextualRelevancyMetric`, which arrives with the retrieval post. Both have relevancy in the name. Answer Relevancy scores the output against the input. Contextual Relevancy scores the retrieved pages against the input. The tell is the field: `actual_output` for one, `retrieval_context` for the other.
**Use when:** the failure mode is padding, deflection or answering a different question.
**Do not use when:** the failure mode is omission. A short, on-topic output that leaves out the important part scores 1.00, as every summary here did.

### FaithfulnessMetric
**Plain meaning:** Does the output say anything that the source it was given does not support?
**Under the hood:** The judge pulls truths from `retrieval_context` and claims from `actual_output`. It marks each claim supported, contradicted or unclear against the truths, and scores the fraction not contradicted. Unclear counts as not contradicted unless `penalize_ambiguous_claims` is set. Four judge calls per test case: truths, claims, verdicts, reason.
**Needs:** `input`, `actual_output` and `retrieval_context`. Referenceless.
**Confused with:** `HallucinationMetric`. Both hunt contradictions. Faithfulness checks the output against `retrieval_context`, the text the system retrieved at runtime. Hallucination checks it against `context`, a ground-truth text set by hand on the test case. Same question, different field.
**Use when:** the system is given text and must not invent beyond it, which is the retrieval post's whole problem.
**Do not use when:** the risk is that the output leaves something out. An omission contradicts nothing.

## Three runs each: the score holds still while the judge does not

Everything above is one run. The previous post showed the assistant producing different summaries from the same ticket on different runs. The judge is the same kind of model, so the same warning applies to the scores. And we have already seen T-1003's summarization score move between two cells. A gap that shows up once is a lead, not a finding. So we score the four money tickets twice more with all three metrics. We report the spread per ticket and per metric. The spread is the highest score minus the lowest across the three runs. We also print the two halves of the summarization score per run. And we print how many truths, claims and statements the judge pulled out each time. A stable score can sit on top of unstable work.

In [9]:
MONEY_CASES = [CASE[t] for t in MONEY_TICKET_IDS]
RUNS = {1: {"Summarization": SUMM, "Answer Relevancy": RELEV, "Faithfulness": FAITH}}
before, t0 = judge_calls(), time.time()
for run in (2, 3):
    RUNS[run] = {"Summarization": await score(SummarizationMetric, MONEY_CASES, n=5),
                 "Answer Relevancy": await score(AnswerRelevancyMetric, MONEY_CASES),
                 "Faithfulness": await score(FaithfulnessMetric, MONEY_CASES)}
repeat_calls = judge_calls() - before

flips = 0
print(f"{'ticket':<8}{'metric':<18}{'run 1':>6}{'run 2':>6}{'run 3':>6}{'spread':>8}  verdict at 0.5")
for tid in MONEY_TICKET_IDS:
    for name in RUNS[1]:
        s = [RUNS[r][name][tid].score for r in (1, 2, 3)]
        verdicts = ["pass" if x >= 0.5 else "fail" for x in s]
        flips += len(set(verdicts)) > 1
        print(f"{tid:<8}{name:<18}{s[0]:>6.2f}{s[1]:>6.2f}{s[2]:>6.2f}{max(s) - min(s):>8.2f}  {' '.join(verdicts)}")

print(f"\nSummarization halves, alignment / coverage, per run")
for tid in MONEY_TICKET_IDS:
    halves = [RUNS[r]["Summarization"][tid].score_breakdown for r in (1, 2, 3)]
    print(f"  {tid}  " + "   ".join(f"{h['Alignment']:.2f} / {h['Coverage']:.2f}" for h in halves))

print(f"\nhow much text the judge pulled out, per run (truths, claims or statements)")
for tid in MONEY_TICKET_IDS:
    parts = []
    for r in (1, 2, 3):
        sm, fm, rm = RUNS[r]["Summarization"][tid], RUNS[r]["Faithfulness"][tid], RUNS[r]["Answer Relevancy"][tid]
        parts.append(f"S {len(sm.truths)}t/{len(sm.claims)}c  F {len(fm.truths)}t/{len(fm.claims)}c  R {len(rm.statements)}s")
    print(f"  {tid}  " + "   ".join(parts))

print(f"\nverdict flipped across runs on {flips} of {4 * 3} ticket-metric pairs")
print(f"judge calls: {repeat_calls} for runs 2 and 3, {time.time() - t0:.0f} s")

ticket  metric             run 1 run 2 run 3  spread  verdict at 0.5
T-1003  Summarization       0.00  0.00  0.00    0.00  fail fail fail
T-1003  Answer Relevancy    1.00  1.00  1.00    0.00  pass pass pass
T-1003  Faithfulness        1.00  1.00  1.00    0.00  pass pass pass
T-1006  Summarization       0.00  0.00  0.00    0.00  fail fail fail
T-1006  Answer Relevancy    1.00  1.00  1.00    0.00  pass pass pass
T-1006  Faithfulness        1.00  1.00  1.00    0.00  pass pass pass
T-1009  Summarization       0.00  0.00  0.00    0.00  fail fail fail
T-1009  Answer Relevancy    1.00  1.00  1.00    0.00  pass pass pass
T-1009  Faithfulness        1.00  1.00  1.00    0.00  pass pass pass
T-1011  Summarization       0.20  0.00  0.20    0.20  fail fail fail
T-1011  Answer Relevancy    1.00  1.00  1.00    0.00  pass pass pass
T-1011  Faithfulness        1.00  1.00  1.00    0.00  pass pass pass

Summarization halves, alignment / coverage, per run
  T-1003  0.00 / 0.20   0.00 / 0.20   0.50 / 0.00


Zero verdict flips across twelve ticket-and-metric pairs. Every relevancy and faithfulness score was 1.00 on all three runs, spread 0.00. Summarization was 0.00 three times on T-1003, T-1006 and T-1009. The only score that moved was T-1011. It went 0.20, 0.00, 0.20 and stayed below the threshold every time. Its alignment half sat exactly on the threshold, at 0.50, on all three runs. On a dashboard, this judge looks rock steady.

In the breakdown, it is not. T-1003's alignment half went 0.00, 0.00, 0.50. T-1009's went 0.33, 0.50, 0.00, and its coverage half went 0.00, 0.00, 0.20. The final score never moved on either ticket. One half was pinned at 0.00 on every run, and the score is the lower half. The number the dashboard shows was stable. The judgment under it was not.

The extraction counts say why. On T-1003 the judge pulled one claim out of the summary on runs 1 and 2 and two claims on run 3. On T-1006 the faithfulness judge found nine truths in the ticket, then six, then nine. Different cuts of the same text give different denominators. Different denominators give different scores. All of this happened under the same temperature setting on every request. The ledger at the end records it. The setting did not buy determinism.

Runs 2 and 3 cost 112 judge calls and 80 seconds. The next cell looks at the two tickets where the movement is easiest to see. T-1009 is the summary that kept the money and still scored zero every time. T-1003 is the ticket whose alignment half moved most.

In [10]:
for run in (1, 2, 3):
    m = RUNS[run]["Summarization"]["T-1009"]
    print(f"run {run}  score {m.score:.2f}  coverage {m.score_breakdown['Coverage']:.2f}")
    for v in m.coverage_verdicts:
        print(f"  ticket {v.original_verdict:<4} summary {v.summary_verdict:<4} {v.question}")
print("\nsummary being scored:", CASE["T-1009"].actual_output)

def alignment(run, tid):
    return RUNS[run]["Summarization"][tid].score_breakdown["Alignment"]

moved = max(MONEY_TICKET_IDS, key=lambda t: max(alignment(r, t) for r in (1, 2, 3)) - min(alignment(r, t) for r in (1, 2, 3)))
print(f"\n{moved}: the claims the judge pulled from its summary, and each verdict, per run")
for run in (1, 2, 3):
    m = RUNS[run]["Summarization"][moved]
    for claim, v in zip(m.claims, m.alignment_verdicts):
        print(f"  run {run}  alignment {m.score_breakdown['Alignment']:.2f}  {v.verdict:<4} {claim}")
print("\nsummary being scored:", CASE[moved].actual_output)

run 1  score 0.00  coverage 0.00
  ticket yes  summary no   Is the team migrating 1,800 customers from their old provider?
  ticket yes  summary no   Do requests start timing out at 30 seconds after about 200 rows?
  ticket yes  summary no   Does the script retry each failed row up to three times?
  ticket yes  summary no   Does the whole import run take hours to complete?
  ticket yes  summary no   Did some imported customers end up with two invoice charges instead of one?
run 2  score 0.00  coverage 0.00
  ticket yes  summary no   Is the team migrating 1,800 customers from their old provider?
  ticket yes  summary no   Do requests start timing out at 30 seconds after about 200 rows?
  ticket yes  summary no   Does the script retry each failed row up to three times?
  ticket yes  summary no   Did the finance team report that about a dozen imported customers have two invoice charges?
  ticket yes  summary no   Can the team clean up the duplicate charges themselves once the import is st

T-1009's summary says the import is "causing duplicate charges". On every run the judge wrote a question about exactly that. Run 1 asked whether some customers "end up with two invoice charges instead of one". Run 2 asked whether "the finance team report that about a dozen imported customers have two invoice charges". Run 3 asked much the same. On every run the judge answered no from the summary. The question carries the ticket's detail: the finance team, the dozen, the invoices. The summary carries none of it. So a summary that kept the one fact the SLA cares about is scored as if it had not. The coverage half rose to 0.20 on run 3 for one reason. A different question, about running fine for 200 rows, happened to be answerable. The money never counted, on the one ticket where the assistant got it right.

T-1003 shows the other half moving. On runs 1 and 2 the judge treated the whole summary as a single claim and marked it "idk", so alignment was 0.00. On run 3 it cut the same sentence into two claims. The second, that the storm followed an outage, was supported, so alignment was 0.50. Nothing about the summary changed. The judge cut the sentence in a different place.

Here is what we take from three runs. The scores were stable because the metric's minimum pinned them at 0.00. Relevancy and faithfulness sat at their ceiling. That is stability from saturation, not from a consistent judge. A metric whose score wanders across its threshold from run to run is what DeepEval calls flaky. Marking it `flaky=True` keeps its score and reason on every report. It stops the metric deciding whether a test case passes. Nothing here crossed the threshold, so nothing needed the flag.

## What teams get wrong

**Treating the score as the output rather than the reason.** T-1003's summarization score moved between runs. No value it took told us anything on its own. The reason did. It said the judge could not find the customer's request among the truths it had extracted. It named the double-charged orders as a question the summary failed. A team that stores the score and discards the reason has kept the wrong half. The score cannot be argued with. The reason can. The reason is one judge call, the cheapest in the metric.

**Comparing scores across judge models.** Every number in this post is one Gemini model's reading of a rubric. Coverage depends on which questions that model chose to write. Alignment depends on where it cut a sentence into claims. A different judge writes different questions and cuts in different places. The same score from two judges is two measurements of different things. They happen to share a scale. A later post on judge agreement puts numbers on how far apart two judges land. Until then, one judge per series. The constant that names it is in the setup cell for that reason.

**Never recording which judge produced a number.** The temperature our wrapper recorded was identical on every call, and it stabilized nothing. The judge model name, the library version, the number of coverage questions and the threshold decided the scores. None of them are in the score. A score with no record of who judged it, on which version, with how many questions, cannot be reproduced. DeepEval prints the evaluation model on every result for that reason. The ledger below records the rest.

In [11]:
usage = [u for u, _ in JUDGE_USAGE]
judge_in = sum(u.prompt_token_count for u in usage)
judge_visible = sum(u.candidates_token_count for u in usage)
judge_thinking = sum(u.thoughts_token_count or 0 for u in usage)
judge_out = judge_visible + judge_thinking
judge_cost = judge_in * PRICE_IN + judge_out * PRICE_OUT

metric_objects = [m for run in RUNS.values() for group in run.values() for m in group.values()] + [summ, quiet]
deepeval_in = sum(m.input_tokens for m in metric_objects)
deepeval_cost = sum(m.evaluation_cost for m in metric_objects)

app_in = sum(u.get("input_tokens", 0) for u in USAGE)
app_out = sum(u.get("output_tokens", 0) for u in USAGE)
app_thinking = sum((u.get("output_token_details") or {}).get("reasoning", 0) for u in USAGE)
app_cost = app_in * PRICE_IN + app_out * PRICE_OUT

per_case = {"Summarization": calls / 12, "Answer Relevancy": relev_calls / 12, "Faithfulness": faith_calls / 12}
print(f"judge calls      {judge_calls()}   per case: " + ", ".join(f"{k} {v:.0f}" for k, v in per_case.items()))
print(f"judge tokens     {judge_in} in / {judge_out} out   (thinking inside 'out': {judge_thinking}, "
      f"{100 * judge_thinking / judge_out:.0f}%)")
print(f"judge cost       ${judge_cost:.4f}   DeepEval's own total: ${deepeval_cost:.4f}, "
      f"{judge_cost / deepeval_cost:.1f} times less   (it counted {deepeval_in} in, matching ours: {deepeval_in == judge_in})")
print(f"app calls        {len(USAGE)}   tokens {app_in} in / {app_out} out   (thinking: {app_thinking})   cost ${app_cost:.4f}")
print(f"total            {judge_calls() + len(USAGE)} model calls, {judge_in + app_in} tokens in / "
      f"{judge_out + app_out} out, thinking {100 * (judge_thinking + app_thinking) / (judge_out + app_out):.0f}% of out, "
      f"${judge_cost + app_cost:.4f}, {time.time() - T0:.0f} s since the setup cell")
print(f"temperatures sent to the judge: {sorted({t for _, t in JUDGE_USAGE})}")

judge calls      293   per case: Summarization 7, Answer Relevancy 3, Faithfulness 4
judge tokens     121678 in / 154718 out   (thinking inside 'out': 138310, 89%)
judge cost       $0.6715   DeepEval's own total: $0.1528, 4.4 times less   (it counted 121678 in, matching ours: True)
app calls        24   tokens 5722 in / 14690 out   (thinking: 14359)   cost $0.0594
total            317 model calls, 127400 tokens in / 169408 out, thinking 90% of out, $0.7308, 241 s since the setup cell
temperatures sent to the judge: [0.0]


Three hundred and seventeen model calls. Twenty-four by the assistant, 293 by the judge. Seven judge calls per case for Summarization, three for Answer Relevancy, four for Faithfulness. All measured, not read from the docs. The judge sent 121,678 tokens in and got 154,718 out. Of those, 138,310 were thinking tokens, 89 percent. The assistant's share looks the way it did in the previous post: 14,359 thinking tokens inside 14,690 out. Across both, thinking was 90 percent of every output token we paid for. Adding a judge did not change the shape of the bill. It multiplied it.

The judge cost $\$0.6715$ at the prices we set. DeepEval's own total for the same calls is $\$0.1528$, 4.4 times less. Its input count matched ours to the token. The gap is entirely thinking tokens it does not see. The whole notebook cost $0.7308 and ran in 241 seconds from the setup cell. That is above the budget we set going in. The overrun is the thinking share. A judge that reasons for a paragraph before answering yes or no is billed for the paragraph. The temperature on every judge call was 0.0, the value DeepEval put there.

## What Halcyon has now, and what it still cannot do

The fingerprint matched, so every number above came from the same corpus as the previous post.

Halcyon now has real scores with reasons. Twelve test cases built from the golden set. Three referenceless metrics wired to a Gemini judge. A measured spread across three runs for every metric on every money ticket. It has a cost ledger that counts thinking tokens, which DeepEval's does not. And it has a blind spot demonstrated rather than asserted. The summary that lost a P1 and the summary that kept one scored the same on all three metrics, on every run.

What it cannot do is express its own P1 rule as a metric. Every metric here compared the output with the input or with the context. None of them read `expected_output`. None of them knew that one fact in the ticket outranks the rest. The rule on the support page says a funds report is P1 regardless of how the ticket is worded. That rule is nowhere in the suite. The suite measures relevance, faithfulness and coverage. The trap is none of those. The trap is a missing sentence that the SLA cares about and the metric does not.

The next post maps the whole DeepEval library and hunts for anything in it that catches what these three missed. A later post on judge agreement asks how much of this spread belongs to the judge and how much to the metric.

Three rules we are taking forward. Decompose every score once before trusting it, because the number was never the finding. Measure the spread before reading a gap, because a single run is a lead. Count the judge's calls and its thinking tokens ourselves, because the framework's counter drops the part that dominates the bill.

### Resources

- DeepEval: Summarization metric: https://deepeval.com/docs/metrics-summarization
- DeepEval: Answer Relevancy metric: https://deepeval.com/docs/metrics-answer-relevancy
- DeepEval: Faithfulness metric: https://deepeval.com/docs/metrics-faithfulness
- DeepEval: metric thresholds, score-only mode and flaky metrics: https://deepeval.com/docs/metrics-introduction
- DeepEval changelog, the higher-is-better breaking change: https://deepeval.com/changelog/changelog-2026#breaking-change
- Introducing DeepEval 4: https://deepeval.com/blog/introducing-deepeval-4
- Gemini API pricing: https://ai.google.dev/gemini-api/docs/pricing
- Latest Gemini models guide, including the removal of sampling parameters: https://ai.google.dev/gemini-api/docs/latest-model
- Companion repository with this notebook and its pyproject.toml: https://github.com/tuhinsharma121/ai-playground/blob/main/eval/deepeval/01-notebook.ipynb